# 🧠 Plan-Execute + Free-Text HIL với Deep Agents SDK

Notebook này demo cơ chế **Plan-Execute với Human-in-the-Loop (HIL) dạng free text** — khác với HIL menu-driven (approve/edit/reject) thông thường.

## Điểm Khác Biệt Chính

| Tính năng | HIL Menu-Driven | HIL Free-Text (notebook này) |
|---|---|---|
| Tương tác | Chọn approve/edit/reject | Gõ bất kỳ yêu cầu nào |
| Agent hiểu | Biết tool bị reject/approve | **Hiểu ý định** từ free text |
| Cập nhật plan | Không (chỉ block/unblock) | **Agent tự gọi `write_todos` lại** để điều chỉnh plan |
| Kích hoạt tool | Tool bị reject không chạy | **Agent có thể gọi tool khác** dựa trên phản hồi |

## Luồng Xử Lý

```
User: "Gửi email marketing cho khách C001, C002 về sản phẩm P002"
       │
       ▼
┌──────────────────────────────────────────────────────────┐
│  1. Agent Plan: gọi write_todos                          │
│     ├─ Task 1: Search C001, C002                         │
│     ├─ Task 2: Search P002                               │
│     ├─ Task 3: Send email campaign                         │
│     └─ Task 4: Generate report                           │
└──────────────────────────────────────────────────────────┘
       │
       ▼ (Execute đến task nguy hiểm → interrupt)
       │
┌──────────────────────────────────────────────────────────┐
│  2. ⛔ HIL Interrupt: "Gửi email đến C001 với nội dung   │
│     ưu đãi 20%, và thêm C003 vào danh sách"              │
└──────────────────────────────────────────────────────────┘
       │
       ▼
┌──────────────────────────────────────────────────────────┐
│  3. Agent HIỂU free text:                                │
│     ├─ "Gửi email C001 với nội dung ưu đãi 20%"          │
│     │   → Cập nhật args của send_email_campaign          │
│     ├─ "thêm C003 vào danh sách"                          │
│     │   → Gọi search_customer(C003) trước khi send email │
│     └─ "và thêm C003" → Gọi write_todos để cập nhật plan │
└──────────────────────────────────────────────────────────┘
       │
       ▼ (Agent execute tiếp với plan đã cập nhật)
```

## Cơ chế

- **interrupt_on**: Chặn tool nguy hiểm (send_email_campaign)
- **Reject + message**: Dùng `reject` decision kèm free text → agent đọc message, hiểu ý định
- **User message injection**: Sau reject, gửi thêm user message → agent tự retry với thông tin mới
- **Re-planning**: Agent tự gọi `write_todos` để cập nhật plan dựa trên HIL feedback


## 1. Cài Đặt Dependencies

In [2]:
# ── Cài đặt dependencies (bỏ comment nếu chạy lần đầu) ──
# import subprocess
# import sys
# deps = [
#     "deepagents>=0.6.0",
#     "langchain-openai>=0.3.0",
#     "langgraph>=0.4.0",
#     "langchain-core>=0.3.0",
#     "python-dotenv>=1.0.0",
# ]
# for dep in deps:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])
# logger.info("✅ All dependencies installed!")

## 2. Import & Khởi Tạo Model

In [3]:
from loguru import logger
import sys

logger.remove()

logger.add(
    sys.stderr,
    format=(
        "<green>{time:DD-MM HH:mm:ss}</green> | "
        # "<level>{level: <5}</level> | "
        "<cyan>{function}</cyan>:<cyan>{line}</cyan>: \n"
        "{message}"
        "\n"
    )
)

1

In [4]:
import os
import json
from datetime import datetime
from typing import Optional

from langchain_core.tools import tool
from langchain_core.messages.ai import AIMessage
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from deepagents import create_deep_agent
from dotenv import load_dotenv

load_dotenv()

# ── Model config ──────────────────────────────────────────────
# Sửa base_url / model theo môi trường của bạn
from langchain_openai import ChatOpenAI

v_glm46 = ChatOpenAI(
    model="v_glm46",
    base_url="https://genai.vnpay.vn/aigateway/llm_glm46/v1",
    temperature=0.7,
    max_tokens=100000,
)

MODEL = v_glm46  # ← Đổi model ở đây nếu cần
logger.info(f"✅ Model configured: {MODEL.model}")

30-06 11:03:16 | <module>:28: 
✅ Model configured: v_glm46



## 3. Định Nghĩa Tools

Các tools mô phỏng nghiệp vụ xử lý yêu cầu:
- `search_customer` / `search_product` — **Safe**, tự động chạy
- `send_email_campaign` — **Nguy hiểm**, bị `interrupt_on` → chờ HIL free text
- `generate_report` — **Safe**, tự động chạy


In [5]:
# ── Mock data ──
CUSTOMERS = {
    "C001": {"name": "Nguyen Van A",  "email": "a@example.com",   "plan": "Premium",   "spent_2026": 25000000},
    "C002": {"name": "Tran Thi B",    "email": "b@example.com",   "plan": "Basic",     "spent_2026": 3500000},
    "C003": {"name": "Le Van C",      "email": "c@example.com",   "plan": "Enterprise","spent_2026": 120000000},
    "C004": {"name": "Pham Thi D",    "email": "d@example.com",   "plan": "Premium",   "spent_2026": 45000000},
}
PRODUCTS = {
    "P001": {"name": "Cloud Storage Basic",   "price": 500000,  "stock": 100},
    "P002": {"name": "Cloud Storage Pro",     "price": 2000000, "stock": 50},
    "P003": {"name": "AI API Access",         "price": 5000000, "stock": 20},
    "P004": {"name": "Enterprise Support",    "price": 10000000,"stock": 10},
}

# ── Safe tools ──
@tool
def search_customer(customer_id: str) -> str:
    """Tra cứu thông tin khách hàng theo ID."""
    result = CUSTOMERS.get(customer_id.upper())
    return json.dumps(result, indent=2, ensure_ascii=False) if result \
           else json.dumps({"error": f"Customer {customer_id} not found"})

@tool
def search_product(product_id: str) -> str:
    """Tra cứu thông tin sản phẩm theo ID."""
    result = PRODUCTS.get(product_id.upper())
    return json.dumps(result, indent=2, ensure_ascii=False) if result \
           else json.dumps({"error": f"Product {product_id} not found"})

@tool
def generate_report(report_type: str, data_summary: str) -> str:
    """Tạo báo cáo tổng kết chiến dịch."""
    return f"📄 Report '{report_type}' generated. Summary: {data_summary[:300]}"

# ── Dangerous tool (cần HIL) ──
@tool
def send_email_campaign(to: str, subject: str, body: str) -> str:
    """Gửi email chiến dịch cho khách hàng. Tool này sẽ tự động chờ người dùng duyệt nội dung."""
    logger.info(f"  📧 Sending campaign email → {to}")
    return f"✅ Campaign email sent to {to}: '{subject}'"

tools = [search_customer, search_product, generate_report, send_email_campaign]
print(f"✅ {len(tools)} tools defined: {[t.name for t in tools]}")
print("   - Safe:      search_customer, search_product, generate_report")
print("   - Dangerous:  send_email_campaign (interrupt_on)")

✅ 4 tools defined: ['search_customer', 'search_product', 'generate_report', 'send_email_campaign']
   - Safe:      search_customer, search_product, generate_report
   - Dangerous:  send_email_campaign (interrupt_on)


In [6]:
print(search_customer.invoke('C001'))

{
  "name": "Nguyen Van A",
  "email": "a@example.com",
  "plan": "Premium",
  "spent_2026": 25000000
}


## 4. Tạo Deep Agent với Free-Text HIL

**Cơ chế Free-Text HIL trong notebook này:**

1. **interrupt_on** chặn `send_email_campaign` → agent dừng chờ duyệt
2. Thay vì menu approve/edit/reject, user nhập **free text** (bất kỳ yêu cầu nào)
3. Agent nhận free text qua **User message** sau khi resume
4. Agent **hiểu ý định** từ free text và tự động:
   - **Cập nhật plan**: gọi `write_todos` để thêm/bớt/sửa task
   - **Kích hoạt tool**: gọi tool khác nếu cần (vd: search thêm customer)
   - **Điều chỉnh args**: sửa nội dung email, thay đổi tham số

> 💡 **Chiến lược System Prompt**: Prompt được thiết kế để agent luôn re-plan khi nhận HIL free text thay đổi yêu cầu.


In [7]:
checkpointer = MemorySaver()

agent = create_deep_agent(
    model=MODEL,
    tools=tools,
    checkpointer=checkpointer,
    interrupt_on={
        "send_email_campaign": {
            "allowed_decisions": ["approve", "edit", "reject"]
        },
    },
    system_prompt="""You are an intelligent customer operations agent.

## CORE BEHAVIOR

### 1. PLAN FIRST
When given a request, ALWAYS call `write_todos` to break it into clear, actionable tasks. Include ALL steps.

### 2. EXECUTE — CALL TOOLS DIRECTLY
When a task needs a tool, call it immediately. Do NOT ask the user for permission — just call the tool. The system will pause automatically for dangerous operations.

### 3. HANDLING HUMAN FEEDBACK (CRITICAL)
After the system resumes from a human review, the user may send additional instructions as free text messages. When you receive such a message:

   a. **ANALYZE** the intent — what does the user want to change?
   b. **RE-PLAN** if needed — call `write_todos` again to update your task list
   c. **EXECUTE** the updated plan — call tools with new parameters

Examples of HIL free text and how to respond:
   - "Thêm C003 vào danh sách, gửi với nội dung ưu đãi 20%"
     → Call search_customer("C003"), then call send_email_campaign with updated args
   - "Bỏ C002, chỉ gửi cho C001 và C003 thôi"
     → Update plan to skip C002, call send_email_campaign for C001 and C003 only
   - "Thay đổi nội dung thành giới thiệu gói Enterprise"
     → Call send_email_campaign with updated body/subject
   - "Trước khi gửi email, hãy search thêm thông tin sản phẩm P004"
     → Call search_product("P004"), then call send_email_campaign
   - "Hủy toàn bộ chiến dịch email, chỉ generate report thôi"
     → Call write_todos to revise plan (remove email tasks, keep report)

### 4. REPORT
After ALL tasks complete, summarize what was done.

Current date: """ + datetime.now().strftime("%Y-%m-%d"),
)

print("✅ Agent created with Free-Text HIL capability!")
print("   → interrupt_on: send_email_campaign (approve/edit/reject)")
print("   → Free-text HIL: gửi user message sau khi resume → agent tự hiểu và re-plan")

✅ Agent created with Free-Text HIL capability!
   → interrupt_on: send_email_campaign (approve/edit/reject)
   → Free-text HIL: gửi user message sau khi resume → agent tự hiểu và re-plan


## 5. Hàm HIL Free-Text Loop

Hàm `run_hil_freetext()` thực hiện toàn bộ vòng lặp:

1. Gọi agent với user request → agent plan + execute
2. Khi gặp interrupt (tool dangerous) → **user nhập free text** bất kỳ
3. Gửi `reject` decision (để block tool hiện tại) kèm free text trong `message`
4. **Gửi thêm user message** với free text → agent hiểu và tự retry (re-plan, gọi tool khác, v.v.)
5. Lặp đến khi hoàn thành


In [8]:
# ! pip install --upgrade loguru

In [54]:
def print_messages(messages, seen_tool_ids=None):
    """In messages ra màn hình dễ đọc (dedup tool calls)."""
    if seen_tool_ids is None:
        seen_tool_ids = set()

    for msg in messages:
        # Tool calls
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                tc_id = tc.get("id", "")
                if tc_id in seen_tool_ids:
                    continue
                seen_tool_ids.add(tc_id)

                if tc["name"] == "write_todos":
                    todos = tc["args"].get("todos", [])
                    logger.info(f"\n  📋 PLAN — {len(todos)} task(s):")
                    for i, t in enumerate(todos, 1):
                        label = t.get('title') or t.get('description') or t.get('content', '?')
                        print(f"       {i}. {label}", flush=True)
                else:
                    args_str = json.dumps(tc["args"], ensure_ascii=False)
                    logger.info(f"\n  🔧 → {tc['name']}({args_str})")

        # Tool results
        if hasattr(msg, "type") and msg.type == "tool":
            logger.info(f"     ✅ Result: {msg.content[:]}")

        # AI text response
        if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
            logger.info(f"\n  🤖 {msg.content[:]}")


def run_hil_freetext(user_input: str, thread_id: str):
    """Vòng lặp Plan-Execute với HIL free text.
    
    Cách hoạt động:
    1. Gọi agent với user_input → agent plan + execute
    2. Nếu interrupt → user gõ free text
    3. Gửi reject(message=free_text) để block tool hiện tại + thông báo
    4. Gửi tiếp user message chứa free_text → agent hiểu, re-plan, retry
    5. Lặp đến khi hoàn thành
    """
    config = {"configurable": {"thread_id": thread_id}}
    seen_tool_ids = set()

    logger.info(f"👤 User: {user_input}")
    logger.info("═" * 70)

    # ── Bước 1: Invoke lần đầu ──
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config,
        version="v2",
    )
    print_messages(result.value.get("messages", []), seen_tool_ids)

    # ── Bước 2: Loop HIL ──
    while result.interrupts:
        iv = result.interrupts[0].value
        action_requests = iv["action_requests"]

        logger.info(f"\n  ⛔ Interrupt: {len(action_requests)} tool(s) cần duyệt")
        for i, action in enumerate(action_requests, 1):
            logger.info(f"     #{i} {action['name']}({json.dumps(action['args'], ensure_ascii=False)})")

        # ── HIL: User nhập FREE TEXT ──
        logger.info(f"\n  ── HIL Free Text ──")
        free_text = input("  ✏️  Phản hồi của bạn (gõ bất kỳ yêu cầu, hoặc 'approve' để duyệt, 'reject' để từ chối): ").strip()

        if free_text.lower() == "approve":
            # Duyệt: cho tool chạy
            decisions = [{"type": "approve"} for _ in action_requests]
            result = agent.invoke(
                Command(resume={"decisions": decisions}),
                config=config,
                version="v2",
            )
        elif free_text.lower() == "reject":
            # Từ chối hoàn toàn
            decisions = [{"type": "reject", "message": "User rejected this operation."} for _ in action_requests]
            result = agent.invoke(
                Command(resume={"decisions": decisions}),
                config=config,
                version="v2",
            )
        else:
            # ── FREE TEXT: reject + gửi user message ──
            # Bước A: Reject tool hiện tại, kèm free text làm message
            decisions = [{"type": "reject", "message": free_text} for _ in action_requests]
            result = agent.invoke(
                Command(resume={"decisions": decisions}),
                config=config,
                version="v2",
            )
            print_messages(result.value.get("messages", []), seen_tool_ids)

            # Bước B: Gửi free text như user message → agent hiểu, re-plan, retry
            logger.info(f"\n  📨 Gửi free text feedback tới agent...")
            result = agent.invoke(
                {"messages": [{"role": "user", "content": free_text}]},
                config=config,
                version="v2",
            )

        print_messages(result.value.get("messages", []), seen_tool_ids)

    # ── Bước 3: Hoàn thành ──
    logger.info("\n" + "═" * 70)
    logger.info("✅ Agent hoàn thành!")
    final_msg = result.value["messages"][-1] if result.value.get("messages") else None
    if final_msg and hasattr(final_msg, "content") and final_msg.content:
        logger.info(f"\n🤖 Final: {final_msg.content[:]}")

    return result, agent


def run_hil_freetext_stream(user_input: str, thread_id: str):
    """Phiên bản stream real-time của HIL free text loop."""
    config = {"configurable": {"thread_id": thread_id}}
    seen_tool_ids = set()
    first = True

    logger.info(f"👤 User: {user_input}")
    logger.info("═" * 70)

    while True:
        if first:
            stream_input = {"messages": [{"role": "user", "content": user_input}]}
            first = False
        else:
            stream_input = Command(resume={"decisions": decisions})

        # Stream
        for event in agent.stream(stream_input, config=config, stream_mode="values", version="v2"):
            msgs = event.get("data", {}).get("messages", [])
            print_messages(msgs, seen_tool_ids)

        # Check state
        state = agent.get_state(config)
        if not state.interrupts:
            break

        # HIL
        iv = state.interrupts[0].value
        action_requests = iv["action_requests"]
        logger.info(f"\n  ⛔ Interrupt: {len(action_requests)} tool(s) cần duyệt")
        for i, action in enumerate(action_requests, 1):
            logger.info(f"     #{i} {action['name']}({json.dumps(action['args'], ensure_ascii=False)})")

        logger.info(f"\n  ── HIL Free Text ──")
        free_text = input("  ✏️  Phản hồi của bạn: ").strip()

        if free_text.lower() == "approve":
            decisions = [{"type": "approve"} for _ in action_requests]
        elif free_text.lower() == "reject":
            decisions = [{"type": "reject", "message": "User rejected."} for _ in action_requests]
        else:
            # Free text: reject + gửi user message
            decisions = [{"type": "reject", "message": free_text} for _ in action_requests]

        # Gửi decisions
        for event in agent.stream(
            Command(resume={"decisions": decisions}),
            config=config,
            stream_mode="values",
            version="v2",
        ):
            msgs = event.get("data", {}).get("messages", [])
            print_messages(msgs, seen_tool_ids)

        # Nếu là free text (không phải approve), gửi thêm user message
        if free_text.lower() not in ("approve", "reject"):
            logger.info(f"\n  📨 Gửi free text feedback tới agent...")
            for event in agent.stream(
                {"messages": [{"role": "user", "content": free_text}]},
                config=config,
                stream_mode="values",
                version="v2",
            ):
                msgs = event.get("data", {}).get("messages", [])
                print_messages(msgs, seen_tool_ids)

            # Kiểm tra interrupt lần nữa (agent có thể gọi tool dangerous khác)
            state = agent.get_state(config)
            if state.interrupts:
                # Vòng lặp while sẽ xử lý
                pass

    logger.info("\n" + "═" * 70)
    logger.info("✅ Agent hoàn thành!")


print("✅ Functions ready!")
print("  📌 run_hil_freetext()           — invoke-based HIL free text loop")
print("  📌 run_hil_freetext_stream()    — stream-based HIL free text loop")

✅ Functions ready!
  📌 run_hil_freetext()           — invoke-based HIL free text loop
  📌 run_hil_freetext_stream()    — stream-based HIL free text loop


## 6. Demo 1 — Free-Text HIL: "Thay đổi nội dung email"

**Scenario:** User yêu cầu gửi email campaign cho khách C001, C002 về sản phẩm P002.  
Khi agent interrupt để duyệt email, user gõ free text để **thay đổi nội dung** thay vì chỉ approve/reject.

> ⚡ Agent sẽ hiểu free text, cập nhật nội dung email và gửi đi.


In [21]:
# ── Demo 1: Thay đổi nội dung email ──
# Bỏ comment dòng dưới và chạy để test

thread_id_1 = str(uuid7())
logger.info(f"🔑 Thread ID: {thread_id_1}")

result , demo_agent = run_hil_freetext(
    user_input="Gửi email marketing cho khách hàng C001 và C002 về sản phẩm P002. Tổng hợp kết quả cuối cùng.",
    thread_id=thread_id_1,
)

# ── Gợi ý free text để test khi interrupt: ──
# ✏️  "Thay đổi nội dung email: thêm ưu đãi 20% cho khách Premium, và thêm C003 vào danh sách"
# ✏️  "Chỉ gửi cho C001 thôi, nội dung tập trung vào tính năng mới"
# ✏️  "approve"  (nếu muốn duyệt luôn)

30-06 11:11:34 | <module>:5: 
🔑 Thread ID: 019f16b9-a8f3-72e3-b5ee-1529eb60839f

30-06 11:11:34 | run_hil_freetext:47: 
👤 User: Gửi email marketing cho khách hàng C001 và C002 về sản phẩm P002. Tổng hợp kết quả cuối cùng.

30-06 11:11:34 | run_hil_freetext:48: 
══════════════════════════════════════════════════════════════════════

30-06 11:11:46 | print_messages:17: 

  📋 PLAN — 5 task(s):



       1. Search customer information for C001 and C002
       2. Search product information for P002
       3. Send email campaign to C001 about P002
       4. Send email campaign to C002 about P002
       5. Generate campaign report


30-06 11:11:46 | print_messages:23: 

  🔧 → search_customer({"customer_id": "C001"})

30-06 11:11:46 | print_messages:23: 

  🔧 → search_customer({"customer_id": "C002"})

30-06 11:11:46 | print_messages:23: 

  🔧 → search_product({"product_id": "P002"})

30-06 11:11:46 | print_messages:27: 
     ✅ Result: Updated todo list to [{'content': 'Search customer information for C001 and C002', 'status': 'in_progress'}, {'content': 'Search product information for P002', 'status': 'pending'}, {'content': 'Send email campaign to C001 about P002', 'status': 'pending'}, {'content': 'Send email campaign to C002 about P002', 'status': 'pending'}, {'content': 'Generate campaign report', 'status': 'pending'}]

30-06 11:11:46 | print_messages:27: 
     ✅ Result: {
  "name": "Nguyen Van A",
  "email": "a@example.com",
  "plan": "Premium",
  "spent_2026": 25000000
}

30-06 11:11:46 | print_messages:27: 
     ✅ Result: {
  "name": "Tran Thi B",
  "email": "b@example.com",
  "plan": "Basic",
  "spent_2026

       1. Search customer information for C001 and C002
       2. Search product information for P002
       3. Send email campaign to C001 about P002
       4. Send email campaign to C002 about P002
       5. Generate campaign report


30-06 11:11:46 | print_messages:23: 

  🔧 → send_email_campaign({"to": "a@example.com", "subject": "Giới thiệu sản phẩm Cloud Storage Pro - Ưu đãi đặc biệt", "body": "Kính gửi Anh Nguyen Van A,\n\nChúng tôi xin giới thiệu sản phẩm Cloud Storage Pro - Giải pháp lưu trữ đám mây chuyên nghiệp với giá chỉ 2.000.000 VNĐ.\n\nSản phẩm phù hợp với khách hàng Premium như Anh, giúp tối ưu hóa việc lưu trữ và quản lý dữ liệu.\n\nLiên hệ ngay để nhận ưu đãi đặc biệt!\n\nTrân trọng,\nĐội ngũ Marketing"})

30-06 11:11:46 | run_hil_freetext:63: 

  ⛔ Interrupt: 1 tool(s) cần duyệt

30-06 11:11:46 | run_hil_freetext:65: 
     #1 send_email_campaign({"to": "a@example.com", "subject": "Giới thiệu sản phẩm Cloud Storage Pro - Ưu đãi đặc biệt", "body": "Kính gửi Anh Nguyen Van A,\n\nChúng tôi xin giới thiệu sản phẩm Cloud Storage Pro - Giải pháp lưu trữ đám mây chuyên nghiệp với giá chỉ 2.000.000 VNĐ.\n\nSản phẩm phù hợp với khách hàng Premium như Anh, giúp tối ưu hóa việc lưu trữ và quản lý dữ liệu.\n\nL

...

30-06 11:11:56 | print_messages:23: 

  🔧 → send_email_campaign({"to": "b@example.com", "subject": "Giới thiệu sản phẩm Cloud Storage Pro - Ưu đãi đặc biệt", "body": "Kính gửi Chị Tran Thi B,\n\nChúng tôi xin giới thiệu sản phẩm Cloud Storage Pro - Giải pháp lưu trữ đám mây chuyên nghiệp với giá chỉ 2.000.000 VNĐ.\n\nSản phẩm phù hợp với khách hàng như Chị, giúp tối ưu hóa việc lưu trữ và quản lý dữ liệu.\n\nLiên hệ ngay để nhận ưu đãi đặc biệt!\n\nTrân trọng,\nĐội ngũ Marketing"})

30-06 11:11:56 | run_hil_freetext:99: 

  📨 Gửi free text feedback tới agent...

30-06 11:12:07 | print_messages:27: 
     ✅ Result: {
  "name": "Nguyen Van A",
  "email": "a@example.com",
  "plan": "Premium",
  "spent_2026": 25000000
}

30-06 11:12:07 | print_messages:27: 
     ✅ Result: Updated todo list to [{'content': 'Search customer information for C001 and C002', 'status': 'in_progress'}, {'content': 'Search product information for P002', 'status': 'pending'}, {'content': 'Send email campaign to C001

       1. Search customer information for C001 and C002
       2. Search product information for P002
       3. Send email campaign to C001 about P002
       4. Send email campaign to C002 about P002
       5. Generate campaign report


30-06 11:12:07 | print_messages:23: 

  🔧 → generate_report({"report_type": "Email Marketing Campaign Summary", "data_summary": "Campaign: Product P002 (Cloud Storage Pro, 2,000,000 VNĐ) sent to C001 (Nguyen Van A, a@example.com, Premium, spent 25M) and C002 (Tran Thi B, b@example.com, Basic, spent 3.5M). Status: Pending user approval for both emails."})

30-06 11:12:07 | print_messages:27: 
     ✅ Result: Updated todo list to [{'content': 'Search customer information for C001 and C002', 'status': 'completed'}, {'content': 'Search product information for P002', 'status': 'completed'}, {'content': 'Send email campaign to C001 about P002', 'status': 'completed'}, {'content': 'Send email campaign to C002 about P002', 'status': 'completed'}, {'content': 'Generate campaign report', 'status': 'in_progress'}]

30-06 11:12:07 | print_messages:27: 
     ✅ Result: 📄 Report 'Email Marketing Campaign Summary' generated. Summary: Campaign: Product P002 (Cloud Storage Pro, 2,000,000 VNĐ) sent to C00

       1. Search customer information for C001 and C002
       2. Search product information for P002
       3. Send email campaign to C001 about P002
       4. Send email campaign to C002 about P002
       5. Generate campaign report


30-06 11:12:07 | print_messages:27: 
     ✅ Result: Updated todo list to [{'content': 'Search customer information for C001 and C002', 'status': 'completed'}, {'content': 'Search product information for P002', 'status': 'completed'}, {'content': 'Send email campaign to C001 about P002', 'status': 'completed'}, {'content': 'Send email campaign to C002 about P002', 'status': 'completed'}, {'content': 'Generate campaign report', 'status': 'completed'}]

30-06 11:12:07 | run_hil_freetext:109: 

══════════════════════════════════════════════════════════════════════

30-06 11:12:07 | run_hil_freetext:110: 
✅ Agent hoàn thành!



In [24]:
for msg in result.value.get('messages'):
    msg.pretty_print()

================================ Human Message =================================

Gửi email marketing cho khách hàng C001 và C002 về sản phẩm P002. Tổng hợp kết quả cuối cùng.
================================== Ai Message ==================================
Tool Calls:
  write_todos (chatcmpl-tool-870f85a9cc14c18b)
 Call ID: chatcmpl-tool-870f85a9cc14c18b
  Args:
    todos: [{'content': 'Search customer information for C001 and C002', 'status': 'in_progress'}, {'content': 'Search product information for P002', 'status': 'pending'}, {'content': 'Send email campaign to C001 about P002', 'status': 'pending'}, {'content': 'Send email campaign to C002 about P002', 'status': 'pending'}, {'content': 'Generate campaign report', 'status': 'pending'}]
  search_customer (chatcmpl-tool-a3d72030ab1265f1)
 Call ID: chatcmpl-tool-a3d72030ab1265f1
  Args:
    customer_id: C001
  search_customer (chatcmpl-tool-9de90a958b48333e)
 Call ID: chatcmpl-tool-9de90a958b48333e
  Args:
    customer_id: C002
  sea

In [25]:

config = {"configurable": {"thread_id": thread_id_1}}


In [26]:
result = demo_agent.invoke(
        {"messages": [{"role": "user", "content": "okie gửi email đi!"}]},
        config=config,
        version="v2",
    )

In [27]:
for msg in result.value.get('messages'):
    msg.pretty_print()

================================ Human Message =================================

Gửi email marketing cho khách hàng C001 và C002 về sản phẩm P002. Tổng hợp kết quả cuối cùng.
================================== Ai Message ==================================
Tool Calls:
  write_todos (chatcmpl-tool-870f85a9cc14c18b)
 Call ID: chatcmpl-tool-870f85a9cc14c18b
  Args:
    todos: [{'content': 'Search customer information for C001 and C002', 'status': 'in_progress'}, {'content': 'Search product information for P002', 'status': 'pending'}, {'content': 'Send email campaign to C001 about P002', 'status': 'pending'}, {'content': 'Send email campaign to C002 about P002', 'status': 'pending'}, {'content': 'Generate campaign report', 'status': 'pending'}]
  search_customer (chatcmpl-tool-a3d72030ab1265f1)
 Call ID: chatcmpl-tool-a3d72030ab1265f1
  Args:
    customer_id: C001
  search_customer (chatcmpl-tool-9de90a958b48333e)
 Call ID: chatcmpl-tool-9de90a958b48333e
  Args:
    customer_id: C002
  sea

In [28]:
result = demo_agent.invoke(
        {"messages": [{"role": "user", "content": "Hãy tổng hợp lại toàn bộ công việc đã làm"}]},
        config=config,
        version="v2",
    )

In [29]:
for msg in result.value.get('messages'):
    msg.pretty_print()

================================ Human Message =================================

Gửi email marketing cho khách hàng C001 và C002 về sản phẩm P002. Tổng hợp kết quả cuối cùng.
================================== Ai Message ==================================
Tool Calls:
  write_todos (chatcmpl-tool-870f85a9cc14c18b)
 Call ID: chatcmpl-tool-870f85a9cc14c18b
  Args:
    todos: [{'content': 'Search customer information for C001 and C002', 'status': 'in_progress'}, {'content': 'Search product information for P002', 'status': 'pending'}, {'content': 'Send email campaign to C001 about P002', 'status': 'pending'}, {'content': 'Send email campaign to C002 about P002', 'status': 'pending'}, {'content': 'Generate campaign report', 'status': 'pending'}]
  search_customer (chatcmpl-tool-a3d72030ab1265f1)
 Call ID: chatcmpl-tool-a3d72030ab1265f1
  Args:
    customer_id: C001
  search_customer (chatcmpl-tool-9de90a958b48333e)
 Call ID: chatcmpl-tool-9de90a958b48333e
  Args:
    customer_id: C002
  sea

In [30]:
result = demo_agent.invoke(
        {"messages": [{"role": "user", "content": "Các email đã được gửi chưa?"}]},
        config=config,
        version="v2",
    )

for msg in result.value.get('messages'):
    msg.pretty_print()

================================ Human Message =================================

Gửi email marketing cho khách hàng C001 và C002 về sản phẩm P002. Tổng hợp kết quả cuối cùng.
================================== Ai Message ==================================
Tool Calls:
  write_todos (chatcmpl-tool-870f85a9cc14c18b)
 Call ID: chatcmpl-tool-870f85a9cc14c18b
  Args:
    todos: [{'content': 'Search customer information for C001 and C002', 'status': 'in_progress'}, {'content': 'Search product information for P002', 'status': 'pending'}, {'content': 'Send email campaign to C001 about P002', 'status': 'pending'}, {'content': 'Send email campaign to C002 about P002', 'status': 'pending'}, {'content': 'Generate campaign report', 'status': 'pending'}]
  search_customer (chatcmpl-tool-a3d72030ab1265f1)
 Call ID: chatcmpl-tool-a3d72030ab1265f1
  Args:
    customer_id: C001
  search_customer (chatcmpl-tool-9de90a958b48333e)
 Call ID: chatcmpl-tool-9de90a958b48333e
  Args:
    customer_id: C002
  sea

In [31]:
result = demo_agent.invoke(
        {"messages": [{"role": "user", "content": "Có"}]},
        config=config,
        version="v2",
    )

for msg in result.value.get('messages'):
    msg.pretty_print()

================================ Human Message =================================

Gửi email marketing cho khách hàng C001 và C002 về sản phẩm P002. Tổng hợp kết quả cuối cùng.
================================== Ai Message ==================================
Tool Calls:
  write_todos (chatcmpl-tool-870f85a9cc14c18b)
 Call ID: chatcmpl-tool-870f85a9cc14c18b
  Args:
    todos: [{'content': 'Search customer information for C001 and C002', 'status': 'in_progress'}, {'content': 'Search product information for P002', 'status': 'pending'}, {'content': 'Send email campaign to C001 about P002', 'status': 'pending'}, {'content': 'Send email campaign to C002 about P002', 'status': 'pending'}, {'content': 'Generate campaign report', 'status': 'pending'}]
  search_customer (chatcmpl-tool-a3d72030ab1265f1)
 Call ID: chatcmpl-tool-a3d72030ab1265f1
  Args:
    customer_id: C001
  search_customer (chatcmpl-tool-9de90a958b48333e)
 Call ID: chatcmpl-tool-9de90a958b48333e
  Args:
    customer_id: C002
  sea

In [43]:
result.interrupts

(Interrupt(value={'action_requests': [{'name': 'send_email_campaign', 'args': {'to': 'a@example.com', 'subject': 'Giới thiệu sản phẩm Cloud Storage Pro - Ưu đãi đặc biệt', 'body': 'Kính gửi Anh Nguyen Van A,\n\nChúng tôi xin giới thiệu sản phẩm Cloud Storage Pro - Giải pháp lưu trữ đám mây chuyên nghiệp với giá chỉ 2.000.000 VNĐ.\n\nSản phẩm phù hợp với khách hàng Premium như Anh, giúp tối ưu hóa việc lưu trữ và quản lý dữ liệu.\n\nLiên hệ ngay để nhận ưu đãi đặc biệt!\n\nTrân trọng,\nĐội ngũ Marketing'}, 'description': "Tool execution requires approval\n\nTool: send_email_campaign\nArgs: {'to': 'a@example.com', 'subject': 'Giới thiệu sản phẩm Cloud Storage Pro - Ưu đãi đặc biệt', 'body': 'Kính gửi Anh Nguyen Van A,\\n\\nChúng tôi xin giới thiệu sản phẩm Cloud Storage Pro - Giải pháp lưu trữ đám mây chuyên nghiệp với giá chỉ 2.000.000 VNĐ.\\n\\nSản phẩm phù hợp với khách hàng Premium như Anh, giúp tối ưu hóa việc lưu trữ và quản lý dữ liệu.\\n\\nLiên hệ ngay để nhận ưu đãi đặc biệt!\\n

In [38]:
type(result.interrupts)

tuple

In [39]:
len(result.interrupts)

1

In [41]:
result.interrupts[0]

Interrupt(value={'action_requests': [{'name': 'send_email_campaign', 'args': {'to': 'a@example.com', 'subject': 'Giới thiệu sản phẩm Cloud Storage Pro - Ưu đãi đặc biệt', 'body': 'Kính gửi Anh Nguyen Van A,\n\nChúng tôi xin giới thiệu sản phẩm Cloud Storage Pro - Giải pháp lưu trữ đám mây chuyên nghiệp với giá chỉ 2.000.000 VNĐ.\n\nSản phẩm phù hợp với khách hàng Premium như Anh, giúp tối ưu hóa việc lưu trữ và quản lý dữ liệu.\n\nLiên hệ ngay để nhận ưu đãi đặc biệt!\n\nTrân trọng,\nĐội ngũ Marketing'}, 'description': "Tool execution requires approval\n\nTool: send_email_campaign\nArgs: {'to': 'a@example.com', 'subject': 'Giới thiệu sản phẩm Cloud Storage Pro - Ưu đãi đặc biệt', 'body': 'Kính gửi Anh Nguyen Van A,\\n\\nChúng tôi xin giới thiệu sản phẩm Cloud Storage Pro - Giải pháp lưu trữ đám mây chuyên nghiệp với giá chỉ 2.000.000 VNĐ.\\n\\nSản phẩm phù hợp với khách hàng Premium như Anh, giúp tối ưu hóa việc lưu trữ và quản lý dữ liệu.\\n\\nLiên hệ ngay để nhận ưu đãi đặc biệt!\\n\

In [42]:
result.interrupts[0].value

{'action_requests': [{'name': 'send_email_campaign',
   'args': {'to': 'a@example.com',
    'subject': 'Giới thiệu sản phẩm Cloud Storage Pro - Ưu đãi đặc biệt',
    'body': 'Kính gửi Anh Nguyen Van A,\n\nChúng tôi xin giới thiệu sản phẩm Cloud Storage Pro - Giải pháp lưu trữ đám mây chuyên nghiệp với giá chỉ 2.000.000 VNĐ.\n\nSản phẩm phù hợp với khách hàng Premium như Anh, giúp tối ưu hóa việc lưu trữ và quản lý dữ liệu.\n\nLiên hệ ngay để nhận ưu đãi đặc biệt!\n\nTrân trọng,\nĐội ngũ Marketing'},
   'description': "Tool execution requires approval\n\nTool: send_email_campaign\nArgs: {'to': 'a@example.com', 'subject': 'Giới thiệu sản phẩm Cloud Storage Pro - Ưu đãi đặc biệt', 'body': 'Kính gửi Anh Nguyen Van A,\\n\\nChúng tôi xin giới thiệu sản phẩm Cloud Storage Pro - Giải pháp lưu trữ đám mây chuyên nghiệp với giá chỉ 2.000.000 VNĐ.\\n\\nSản phẩm phù hợp với khách hàng Premium như Anh, giúp tối ưu hóa việc lưu trữ và quản lý dữ liệu.\\n\\nLiên hệ ngay để nhận ưu đãi đặc biệt!\\n\\n

In [50]:
act_requests = result.interrupts[0].value['action_requests']
rev_configs = result.interrupts[0].value['review_configs']

rev_configs, act_requests

([{'action_name': 'send_email_campaign',
   'allowed_decisions': ['approve', 'edit', 'reject']},
  {'action_name': 'send_email_campaign',
   'allowed_decisions': ['approve', 'edit', 'reject']}],
 [{'name': 'send_email_campaign',
   'args': {'to': 'a@example.com',
    'subject': 'Giới thiệu sản phẩm Cloud Storage Pro - Ưu đãi đặc biệt',
    'body': 'Kính gửi Anh Nguyen Van A,\n\nChúng tôi xin giới thiệu sản phẩm Cloud Storage Pro - Giải pháp lưu trữ đám mây chuyên nghiệp với giá chỉ 2.000.000 VNĐ.\n\nSản phẩm phù hợp với khách hàng Premium như Anh, giúp tối ưu hóa việc lưu trữ và quản lý dữ liệu.\n\nLiên hệ ngay để nhận ưu đãi đặc biệt!\n\nTrân trọng,\nĐội ngũ Marketing'},
   'description': "Tool execution requires approval\n\nTool: send_email_campaign\nArgs: {'to': 'a@example.com', 'subject': 'Giới thiệu sản phẩm Cloud Storage Pro - Ưu đãi đặc biệt', 'body': 'Kính gửi Anh Nguyen Van A,\\n\\nChúng tôi xin giới thiệu sản phẩm Cloud Storage Pro - Giải pháp lưu trữ đám mây chuyên nghiệp vớ

In [52]:
decisions = [
    {
        'type': 'approve'
    },
    {
        'type': 'reject',
        'message': 'User rejected this operation.'
    }
]

result = demo_agent.invoke(
        Command(resume = {"decisions": decisions}),
        config=config,
        version="v2",
    )

for msg in result.value.get('messages'):
    msg.pretty_print()

30-06 13:51:36 | send_email_campaign:39: 
  📧 Sending campaign email → a@example.com



================================ Human Message =================================

Gửi email marketing cho khách hàng C001 và C002 về sản phẩm P002. Tổng hợp kết quả cuối cùng.
================================== Ai Message ==================================
Tool Calls:
  write_todos (chatcmpl-tool-870f85a9cc14c18b)
 Call ID: chatcmpl-tool-870f85a9cc14c18b
  Args:
    todos: [{'content': 'Search customer information for C001 and C002', 'status': 'in_progress'}, {'content': 'Search product information for P002', 'status': 'pending'}, {'content': 'Send email campaign to C001 about P002', 'status': 'pending'}, {'content': 'Send email campaign to C002 about P002', 'status': 'pending'}, {'content': 'Generate campaign report', 'status': 'pending'}]
  search_customer (chatcmpl-tool-a3d72030ab1265f1)
 Call ID: chatcmpl-tool-a3d72030ab1265f1
  Args:
    customer_id: C001
  search_customer (chatcmpl-tool-9de90a958b48333e)
 Call ID: chatcmpl-tool-9de90a958b48333e
  Args:
    customer_id: C002
  sea

In [53]:
result = demo_agent.invoke(
        {"messages": [{'role': 'user', 'content': 'đã gửi email cho khách hàng nào và khách hàng nào bị từ chối gửi email?'}]},
        config=config,
        version="v2",
    )

for msg in result.value.get('messages'):
    msg.pretty_print()

================================ Human Message =================================

Gửi email marketing cho khách hàng C001 và C002 về sản phẩm P002. Tổng hợp kết quả cuối cùng.
================================== Ai Message ==================================
Tool Calls:
  write_todos (chatcmpl-tool-870f85a9cc14c18b)
 Call ID: chatcmpl-tool-870f85a9cc14c18b
  Args:
    todos: [{'content': 'Search customer information for C001 and C002', 'status': 'in_progress'}, {'content': 'Search product information for P002', 'status': 'pending'}, {'content': 'Send email campaign to C001 about P002', 'status': 'pending'}, {'content': 'Send email campaign to C002 about P002', 'status': 'pending'}, {'content': 'Generate campaign report', 'status': 'pending'}]
  search_customer (chatcmpl-tool-a3d72030ab1265f1)
 Call ID: chatcmpl-tool-a3d72030ab1265f1
  Args:
    customer_id: C001
  search_customer (chatcmpl-tool-9de90a958b48333e)
 Call ID: chatcmpl-tool-9de90a958b48333e
  Args:
    customer_id: C002
  sea

## 7. Demo 2 — Free-Text HIL: "Hủy email, chuyển sang gọi điện"

**Scenario:** User yêu cầu gửi email marketing.  
Khi agent interrupt, user gõ free text yêu cầu **thay đổi hoàn toàn chiến lược**: không gửi email nữa, mà chuyển sang gọi điện cho khách hàng VIP.

> ⚡ Agent hiểu ý định "hủy email, chuyển sang gọi điện" → re-plan → gọi tools khác (search_customer, generate_report) thay vì send_email_campaign.


In [25]:
# ── Demo 2: Hủy email, chuyển sang gọi điện ──
# Bỏ comment dòng dưới và chạy để test

thread_id_2 = str(uuid7())
logger.info(f"🔑 Thread ID: {thread_id_2}")

run_hil_freetext(
    user_input="Gửi email marketing cho khách hàng C001 về sản phẩm P003 (AI API Access).",
    thread_id=thread_id_2,
)

# ── Gợi ý free text để test khi interrupt: ──
# ✏️  "Không gửi email nữa. Hãy search thông tin C001 và tạo báo cáo tổng quan về khách hàng này."
# ✏️  "Thay vì email, hãy search thêm sản phẩm P004 và generate report so sánh P003 vs P004"

29-06 17:18:33 | INFO  | <module>:5 | 🔑 Thread ID: 019f12e3-4993-7370-913e-b9a6e9eb9bf9
29-06 17:18:33 | INFO  | run_hil_freetext:47 | 👤 User: Gửi email marketing cho khách hàng C001 về sản phẩm P003 (AI API Access).
29-06 17:18:33 | INFO  | run_hil_freetext:48 | ══════════════════════════════════════════════════════════════════════
29-06 17:18:37 | INFO  | print_messages:23 | 
  🔧 → search_customer({"customer_id": "C001"})
29-06 17:18:37 | INFO  | print_messages:23 | 
  🔧 → search_product({"product_id": "P003"})
29-06 17:18:37 | INFO  | print_messages:27 |      ✅ Result: {
  "name": "Nguyen Van A",
  "email": "a@example.com",
  "plan": "Premium",
  "spent_2026": 25000000
}
29-06 17:18:37 | INFO  | print_messages:27 |      ✅ Result: {
  "name": "AI API Access",
  "price": 5000000,
  "stock": 20
}
29-06 17:18:37 | INFO  | print_messages:23 | 
  🔧 → send_email_campaign({"body": "Chào anh Nguyen Van A,\n\nChúng tôi xin giới thiệu đến anh sản phẩm AI API Access - giải pháp API mạnh mẽ dành

## 8. Demo 3 — Stream version (real-time output)

Phiên bản **stream real-time** của HIL free text loop — bạn thấy agent đang làm gì trong thời gian thực thay vì chờ kết quả cuối cùng.


In [55]:
# ── Demo 3: Stream version ──
# Bỏ comment dòng dưới và chạy để test

thread_id_3 = str(uuid7())
logger.info(f"🔑 Thread ID: {thread_id_3}")

run_hil_freetext_stream(
    user_input="Gửi email chào mừng cho khách hàng C003 về gói Enterprise.",
    thread_id=thread_id_3,
)

30-06 14:02:33 | <module>:5: 
🔑 Thread ID: 019f1756-3477-7991-bd8d-56ee612eae2b

30-06 14:02:33 | run_hil_freetext_stream:124: 
👤 User: Gửi email chào mừng cho khách hàng C003 về gói Enterprise.

30-06 14:02:33 | run_hil_freetext_stream:125: 
══════════════════════════════════════════════════════════════════════

30-06 14:02:35 | print_messages:23: 

  🔧 → search_customer({"customer_id": "C003"})

30-06 14:02:35 | print_messages:27: 
     ✅ Result: {
  "name": "Le Van C",
  "email": "c@example.com",
  "plan": "Enterprise",
  "spent_2026": 120000000
}

30-06 14:02:37 | print_messages:27: 
     ✅ Result: {
  "name": "Le Van C",
  "email": "c@example.com",
  "plan": "Enterprise",
  "spent_2026": 120000000
}

30-06 14:02:37 | print_messages:23: 

  🔧 → send_email_campaign({"to": "c@example.com", "subject": "Chào mừng bạn đến với gói Enterprise!", "body": "Kính gửi Anh Le Van C,\n\nChúc mừng Anh đã trở thành thành viên của gói Enterprise!\n\nChúng tôi rất hân hạnh được đồng hành cùng Anh tr

## 9. Tổng Kết

### Cơ chế Free-Text HIL hoạt động như thế nào?

```
User free text: "Thêm C003, đổi nội dung thành ưu đãi 20%"
       │
       ├─▶ Step 1: Reject tool + message="Thêm C003, đổi nội dung thành ưu đãi 20%"
       │         → Agent thấy: tool bị reject kèm lý do
       │
       └─▶ Step 2: User message="Thêm C003, đổi nội dung thành ưu đãi 20%"
                 → Agent hiểu ý định:
                    • "Thêm C003" → search_customer("C003")
                    • "đổi nội dung thành ưu đãi 20%" → send_email_campaign with new body
                    • Có thể gọi write_todos để re-plan
```

### Khi nào dùng Free-Text HIL?

| Tình huống | Lý do |
|---|---|
| Thay đổi nội dung tool | User muốn sửa email body/subject/args |
| Thêm bước mới | User muốn search thêm trước khi execute |
| Thay đổi chiến lược | User muốn hủy email, chuyển sang gọi điện |
| Điều chỉnh phạm vi | User muốn thêm/bớt khách hàng, sản phẩm |
| Phản hồi phức tạp | User có nhiều yêu cầu cùng lúc |

### So sánh với HIL Menu-Driven

| Tiêu chí | Menu-Driven | Free-Text (notebook này) |
|---|---|---|
| UX | Chọn từ danh sách | Gõ tự do |
| Agent hiểu | Chỉ biết tool được approve hay không | **Hiểu ngữ cảnh, ý định** |
| Re-plan | Không | ✅ Agent tự gọi write_todos |
| Kích hoạt tool mới | Không | ✅ Agent gọi tool khác dựa trên HIL |
| Độ phức tạp | Thấp, dễ implement | Cao hơn, cần prompt engineering |
